In [1]:
#!pip install -r requirements.txt

In [2]:
import os
import time
import json

import pandas as pd

In [3]:
from src.process_function import process_transaction
from src.fraud_detection_service.embedding_rule_utils import load_and_embed_rules
from src.transaction import Transaction

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

KeyboardInterrupt: 

In [ ]:
# LLM Configuration
os.environ['llm_model_name'] = 'llama3.1'
os.environ['llm_model_temperature'] = '0.5'
# RAG Configuration
os.environ['rag_model_name'] = 'llama3.1'
# Chat Model Configuration
os.environ['chat_model_name'] = 'llama3.1'
os.environ['embedding_model_name'] = 'nomic-embed-text'

In [ ]:
BANK_TRANSACTIONS = os.path.join(os.path.abspath(os.path.dirname(os.path.dirname(os.path.dirname('main_app.py')))), 'data', 'bank_transactions_data_2.csv')
FRAUD_RULES = os.path.join(os.path.abspath(os.path.dirname(os.path.dirname(os.path.dirname('main_app.py')))), 'data', 'fraud_rules.json')

In [ ]:
df = pd.read_csv(BANK_TRANSACTIONS)

In [ ]:
processed_transactions = []
output_filename = "processed_transactions_results.jsonl"

In [ ]:
embedding_model_name = os.environ.get('embedding_model_name', 'all-minilm')
embedded_rules = load_and_embed_rules(model_name=embedding_model_name, rules_file_path=FRAUD_RULES)

In [ ]:
# Delete the output file if it exists
if os.path.exists(output_filename):
    os.remove(output_filename)

In [ ]:
prompt_template = {
    'system_prompt': lambda _: (
            "Eres un experto en detección de fraude bancario. Tu tarea es analizar los detalles de una transacción "
            "y determinar si es potencialmente fraudulenta. Responde SÓLO con 'FRAUDULENTO' o 'NORMAL'."
            "Considera indicadores como montos inusuales, ubicaciones geográficas sospechosas, múltiples intentos de login,"
            "dispositivos o canales no reconocidos, o transacciones que vacían la cuenta."
        ),
    'user_prompt': lambda transaction_details_for_llm: (
            f"Analiza la siguiente transacción:\n\n"
            f"{transaction_details_for_llm}\n\n"
            f"Basado en los detalles proporcionados, ¿es esta transacción FRAUDULENTA o NORMAL? "
            f"Responde SÓLO con 'FRAUDULENTO' o 'NORMAL'."
        ),
    'explainer': lambda transaction_details_str, rules_text: (
            f"Eres un analista experto en riesgo financiero y detección de anomalías transaccionales.\n"
            f"Tu tarea es evaluar la siguiente transacción que ha sido marcada como **de alto riesgo** o **anómala**.\n"
            f"Basándote en los detalles proporcionados y las reglas de comportamiento sospechoso, genera una explicación concisa del **porqué esta transacción es considerada anómala o potencialmente de riesgo**.\n"
            f"Incluye el razonamiento basado en las reglas aplicables y sugiere las **acciones de mitigación** más relevantes.\n\n"
            f"**Detalles de la Transacción**:\n{transaction_details_str}\n\n"
            f"**Reglas de comportamiento anómalo relevantes encontradas**:\n{rules_text}\n\n"
        )
}

In [ ]:
print(f"Iniciando procesamiento de {len(df)} transacciones...")
for i, transaction_data in enumerate(df.to_dict(orient='records')):
    start_time = time.time()
    print(f"Procesando transacción {i+1}/{len(df)}: ID {transaction_data.get('TransactionID')}")
    try:
        result = process_transaction(transaction_data=transaction_data, embedded_rules=embedded_rules, prompt_template=prompt_template)
        processed_transactions.append(result)

        with open(output_filename, 'a', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False)
            f.write('\n')

    except Exception as e:
        print(f"Error al procesar transacción {transaction_data.get('TransactionID')}: {e}")
        processed_transactions.append({**transaction_data, 'error': str(e)})
    if i % 10 == 0:
        print(f"Progreso: {i+1}/{len(df)} transacciones procesadas.")
    # Terminar el bucle si se alcanza un número específico de transacciones
    # cuando i sea 100 terminar el bucle
    elapsed_time = time.time() - start_time
    print(f"Tiempo transcurrido para la transacción {i+1}: {elapsed_time:.2f} segundos")
    
    if i >= 100:
        print("Procesamiento interrumpido después de 100 transacciones.")
        break
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")

        

print(f"Procesamiento completado. Resultados guardados en {output_filename}.")

In [ ]:
# contar el nuero de transacciones procesadas
processed_count = len(processed_transactions)
print(f"Número total de transacciones procesadas: {processed_count}")
# contar el numero de transacciones fraudulentas
fraudulent_count = sum(1 for t in processed_transactions if t.get('is_fraud', False))
print(f"Número total de transacciones fraudulentas: {fraudulent_count}")
# contar el numero de transacciones no fraudulentas
non_fraudulent_count = processed_count - fraudulent_count
print(f"Número total de transacciones no fraudulentas: {non_fraudulent_count}")
# contar las transacciones fraudulentas por tags de fraude_rules
fraudulent_tags = {}
for transaction in processed_transactions:
    if transaction.get('is_fraud', False):
        tags = transaction.get('fraud_tags', [])
        for tag in tags:
            if tag not in fraudulent_tags:
                fraudulent_tags[tag] = 0
            fraudulent_tags[tag] += 1
# contar las transacciones no fraudulentas por tags de fraude_rules
non_fraudulent_tags = {}
for transaction in processed_transactions:
    if not transaction.get('is_fraud', False):
        tags = transaction.get('fraud_tags', [])
        for tag in tags:
            if tag not in non_fraudulent_tags:
                non_fraudulent_tags[tag] = 0
            non_fraudulent_tags[tag] += 1
# imprimir los tags de fraude_rules
print("Tags de fraude_rules:")
for tag, count in fraudulent_tags.items():
    print(f"{tag}: {count} transacciones fraudulentas")
print("Tags de fraude_rules no fraudulentas:")
for tag, count in non_fraudulent_tags.items():
    print(f"{tag}: {count} transacciones no fraudulentas")